# 정기 repository reviewer 만들기

## 소개

반복되는 review 수고를 직접 제어하는 무인 작업으로 바꾸세요. 잘 검토된 repository도 검토 사이에 변경되며 이전 review가 지적하지 않은 문제가 들어올 수 있습니다. 이 예제는 그 일을 agent에 맡깁니다.

선택한 일정에 따라 agent는 repository를 훑고 프로그램이 읽을 수 있는 판정과 발견을 반환합니다. 실행 간 연속성은 이 예제의 중심인 Agent SDK의 재개 세션 기능에서 나옵니다. 재개는 정기 reviewer의 설계 선택 하나입니다. 재개 세션은 별도 store를 만들지 않고 이전 검토 전체를 이어받습니다. 각 실행은 빈 상태 대신 발견과 reviewer가 읽은 모든 내용이 이미 불러와진 상태로 시작합니다. *언제 재개하고 언제 새로 시작할지* 절은 선택별로 적합한 작업을 설명합니다.

repository는 첫 실행에서 전체 baseline review를 받습니다. 이후 모든 실행은 이전 실행의 세션을 재개하고 이미 보고한 발견에서 시작합니다. 매 주기마다 짧은 후속 결과를 읽고, 각 후속 결과는 이전 review의 발견을 되풀이하여 연속성을 증명합니다. 이 notebook은 첫 두 주기를 수동 실행하고 같은 reviewer를 정기 실행하도록 설정하며 끝납니다.

## 배울 내용

이 예제를 마치면 다음을 할 수 있습니다.

- scheduler용 `max_turns`와 `max_budget_usd`가 설정된 `permission_mode="dontAsk"`에서 한도가 있는 read-only review agent를 무인 실행하기
- `ClaudeAgentOptions(resume=...)`로 재개하고 `output_format` schema 응답의 `RESUME-LINK` 필드를 assertion으로 확인하여 실행 간 연속성 증명하기. 수정된 발견은 `resolved`로 이동하고 새로 심은 버그는 잡습니다.
- `scheduled_review.py`, grep 가능한 `VERDICT` 및 완료 줄, 0이 아닌 종료 코드로 끝나는 좁은 `except ResultError` 경로로 reviewer를 cron에 넣기

## 이 예제를 사용할 때

dependency audit나 문서 최신성 점검처럼 이전 검토를 기억하는 반복 review에 사용하세요. scheduler는 직접 실행하며 Claude Code가 읽기 tool, permission 규칙, 재개 가능한 세션을 제공합니다.

scheduler나 infrastructure를 운영하고 싶지 않다면 다음 managed 선택지가 더 적합할 수 있습니다.

- [Claude Code routine](https://code.claude.com/docs/en/routines)은 자신의 GitHub repository를 정기적으로 검토하고 싶은 개인 개발자에게 적합합니다. prompt, repository, connector를 한 번 설정하면 Claude가 managed infrastructure에서 routine을 실행합니다. routine은 research preview이며 Claude 구독 plan에서 실행됩니다.
- [Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview)는 Claude API로 설정하는 managed hosting을 원할 때 적합합니다. agent 루프와 workspace를 host하고 [scheduled deployment](https://platform.claude.com/docs/en/managed-agents/scheduled-deployments)가 설정한 cron 주기로 세션을 시작합니다. Managed Agents는 beta이며 검토 대상 repository는 hosted 세션 workspace에 들어갑니다.

scheduler와 review environment를 직접 운영하고 싶다면 이 예제를 사용하세요.


## 사전 요구 사항

이 guide를 따르기 전에 다음을 준비하세요.

**필요한 지식:**

* Python 기초: async/await, 함수, 기본 자료 구조를 편하게 사용
* agentic 패턴의 기본 이해: [효과적인 agent 만들기](https://www.anthropic.com/engineering/building-effective-agents)를 읽기를 권장합니다.
* Agent SDK의 `query()` 함수에 대한 기본 이해: SDK가 처음이라면 [한 줄 research agent](../00_The_one_liner_research_agent.ipynb)를 참고하세요.

**필요한 도구:**

* Python 3.11 이상
* claude-agent-sdk 0.2.140 이상. 이 예제의 실패 경로에서 잡는 타입이 지정된 `ResultError`를 추가한 release입니다.
* Anthropic API key([여기서 발급](https://platform.claude.com/settings/keys))

작성하는 파일이 옆에 놓이도록 이 notebook 자체 디렉터리에서 실행하세요.

**비용 참고:** 이 notebook의 실제 실행 전체는 API 사용량으로 보통 약 10센트가 들며 동반 script 실행도 비슷합니다. 각 실행은 요약 줄에 자체 비용을 출력합니다.

## 설정

먼저 필요한 dependency를 설치하세요.


In [1]:
%%capture
%pip install -U "claude-agent-sdk>=0.2.140" python-dotenv

**참고:** `.env`에 다음이 있는지 확인하세요.

```bash
ANTHROPIC_API_KEY=your_key_here
```

environment variable을 불러옵니다.


In [2]:
from dotenv import load_dotenv

_ = load_dotenv()

## 검토할 예제 repository 만들기

reviewer의 보고를 확인할 수 있도록 알려진 결함이 있는 repository를 만듭니다. 아래 코드는 이 notebook의 review 대상인 작은 서비스를 `demo_repo/`에 만듭니다. 각 파일의 source는 코드가 디스크에 쓰는 문자열 상수로 나타납니다. 파일을 import하지 않으므로 버그는 실행되지 않으며, 마지막 정리 절에서 완료 후 `demo_repo/`를 제거합니다. 서비스의 두 애플리케이션 파일은 각각 실제 버그 하나를 가집니다.

* `app/config.py`는 전체 environment를 복사하여 config를 만든 뒤 결과를 출력합니다. 두 줄이 함께 process가 가진 모든 environment variable을 secret까지 log에 남깁니다.
* `app/math_utils.py`는 `len(values)`로 나누어 평균을 구합니다. 빈 목록을 전달하면 `ZeroDivisionError`가 발생합니다.


In [3]:
from pathlib import Path

REPO_DIR = Path("demo_repo").resolve()

README_MD = """# sample-repo

A small service kept here so the scheduled reviewer has something to read.
"""

CONFIG_PY = """import os


def load_config(path):
    config = {"source": path}
    config["environment"] = dict(os.environ)
    print("loaded config:", config)
    return config
"""

MATH_UTILS_PY = """def divide(numerator, denominator):
    return numerator / denominator


def average(values):
    return divide(sum(values), len(values))
"""

SAMPLE_REPO_FILES = {
    "README.md": README_MD,
    "app/config.py": CONFIG_PY,
    "app/math_utils.py": MATH_UTILS_PY,
}

for relative_path, body in SAMPLE_REPO_FILES.items():
    target = REPO_DIR / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(body, encoding="utf-8")

seeded = sorted(str(path.relative_to(REPO_DIR)) for path in REPO_DIR.rglob("*") if path.is_file())
print("sample repo:", seeded)

sample repo: ['README.md', 'app/config.py', 'app/math_utils.py']


## review가 반환하는 내용 정의하기

예제 repository가 준비되면 다음은 review의 응답 contract입니다. 정기 review를 소비하는 것은 모두 프로그램이므로 reviewer는 필드별로 읽는 JSON 객체로 답합니다. `output_format`은 JSON schema를 받으며 응답은 이미 그 형태인 `ResultMessage.structured_output`으로 도착합니다. 각 review는 review id, 닫힌 집합의 판정, 각각 자체 id를 가진 발견을 담습니다.

발견 id가 연속성 증명을 이끕니다. 후속 review는 이전 실행의 id를 명시하여 발견을 이어받았음을 보여 줍니다.


In [4]:
from enum import StrEnum


class Verdict(StrEnum):
    """정기 review가 보고할 수 있는 두 판정."""

    OK = "ok"
    CONCERNS = "concerns"


VERDICT_VALUES = [verdict.value for verdict in Verdict]

FINDING_SCHEMA = {
    "type": "object",
    "properties": {
        "id": {"type": "string"},
        "file": {"type": "string"},
        "summary": {"type": "string"},
    },
    "required": ["id", "file", "summary"],
}

FIRST_REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "review_id": {"type": "string"},
        "verdict": {"type": "string", "enum": VERDICT_VALUES},
        "findings": {"type": "array", "items": FINDING_SCHEMA},
    },
    "required": ["review_id", "verdict", "findings"],
}

## 후속 review용 schema 확장하기

이 예제는 review 두 번을 실행하며 두 번째에는 첫 번째를 가리킬 방법이 필요합니다. `previous_review_id`와 `previous_finding_ids`가 첫 review를 가리키며 `resolved`는 더 이상 적용되지 않는 이전 발견 id를 나열합니다. 첫 schema를 merge하면 공유 부분을 한곳에 유지하므로 둘이 어긋날 수 없습니다.

`previous_review_id`와 `previous_finding_ids`는 연속성 assertion을 담습니다. reviewer가 이전 review를 이어받으면 필드는 첫 실행의 review id와 발견 id를 담아 돌아옵니다. 처음부터 다시 시작하면 필드가 그렇게 나타내며 실행 log 줄로 알 수 있습니다.


In [5]:
CONTINUITY_PROPERTIES = {
    "previous_review_id": {"type": "string"},
    "previous_finding_ids": {"type": "array", "items": {"type": "string"}},
    "resolved": {"type": "array", "items": {"type": "string"}},
}

FOLLOW_UP_REVIEW_SCHEMA = FIRST_REVIEW_SCHEMA | {
    "properties": FIRST_REVIEW_SCHEMA["properties"] | CONTINUITY_PROPERTIES,
    "required": [*FIRST_REVIEW_SCHEMA["required"], *CONTINUITY_PROPERTIES.keys()],
}

print("follow-up required fields:", FOLLOW_UP_REVIEW_SCHEMA["required"])

follow-up required fields: ['review_id', 'verdict', 'findings', 'previous_review_id', 'previous_finding_ids', 'resolved']


## 한도가 있는 read-only reviewer 설정하기

두 schema를 정의했으므로 다음은 agent 설정입니다. 이 옵션은 무인 실행 agent에 적합합니다.

* **`tools`**: agent에 어떤 built-in tool이 존재할지 결정합니다. `Read`, `Glob`, `Grep`로 제한하면 쓰기 tool이 없습니다.
* **`allowed_tools`**: permission prompt 없이 실행할 호출을 결정합니다. `Read` grant는 repository 경로로 범위가 제한되며 `dontAsk`에서는 바깥 읽기가 tool 오류로 거부됩니다. `Glob`과 `Grep`는 전체 tool을 승인하는 단순 grant로 들어가고 `deny_reads_outside_repo` hook이 범위를 제한합니다. 실행 요약의 `StructuredOutput` 항목은 이 목록 대신 `output_format`에서 나오며 응답만 담습니다.
* **`permission_mode="dontAsk"`**: 절대 입력을 묻지 않고 사전 승인되지 않은 것을 거부합니다. `auto` permission mode도 일상적인 prompt 없이 실행하며 runtime model classifier가 각 호출을 승인하거나 거부합니다. 이 reviewer는 tool 인터페이스를 허용 규칙으로 고정하기 위해 `dontAsk`를 사용합니다.
* **`model`**: review를 지정한 model로 고정합니다. 고정하지 않으면 SDK는 시간이 지나며 바뀔 수 있는 environment의 기본 model을 사용합니다. 고정은 변경하기로 결정할 때까지 정기 작업의 비용과 review 동작을 설정한 곳에 유지합니다.
* **`strict_mcp_config=True`**: 이 설정이 선언한 MCP server로 세션을 제한하며 여기서는 없습니다. 없으면 기기나 계정에 설정된 MCP server가 세션에 연결되고 tool 정의가 모든 요청에 들어갑니다.
* **`max_turns`와 `max_budget_usd`**: agentic 루프와 지출을 제한합니다. 둘 중 하나를 넘으면 terminal 오류 결과인 `error_max_turns` 또는 `error_max_budget_usd`로 끝나며 SDK는 코드에 `ResultError`로 발생시킵니다. 지출이 budget을 이미 넘은 후에 실행이 중단되므로 주기가 상한을 초과한 비용으로 끝날 수 있습니다.
* **`hooks`**: `deny_reads_outside_repo`를 [`PreToolUse`](https://code.claude.com/docs/en/agent-sdk/hooks) callback으로 등록합니다. callback은 모든 `Read`, `Grep`, `Glob` 호출을 보고 경로가 repository 밖으로 결정되면 거부하며, `/`나 `~`로 시작하거나 `..` segment 또는 `{` brace를 담은 Glob 패턴도 거부합니다.
* **`setting_sources=[]`**: 실행이 기기의 filesystem 설정을 사용하지 않게 합니다. 정기 review는 laptop과 cron 실행 기기에서 같게 동작합니다. 빈 목록은 검토 대상 repository의 `.claude/settings.json` 및 `CLAUDE.md`도 세션에서 제외합니다. reviewer의 `cwd`는 해당 repository이며 직접 제어하지 않는 repository가 자신의 reviewer를 설정해서는 안 됩니다.
* **`output_format`과 `resume`**: `output_format`은 전달한 schema에 맞는 응답을 요청하고 `resume`은 실행 간 연속성을 유지합니다. 최초 실행에서는 `None`, 정기 후속 실행에서는 이전 실행의 세션 id를 전달합니다.

### 이 옵션이 구현하는 보호 조치

위 옵션은 agent 무인 실행의 여러 보호 조치를 구현합니다. read-only tool grant, MCP 및 설정 격리, turn 및 budget 상한은 실행 가능한 범위를 제한합니다. 정기적인 세션 reset은 context의 지속 시간을 제한합니다.

읽기는 두 번 범위를 제한합니다. notebook과 script 모두 `allowed_tools` grant가 `Read`를 repository 경로에 제한하며 읽기는 작업 사용자 자체의 파일 접근도 따릅니다. review에 필요한 접근으로 제한된 사용자로 deployment 단계를 실행하세요.

Claude Code는 `Grep` 및 `Glob`에 `Read` 규칙을 best-effort로 적용합니다. `deny_reads_outside_repo`는 이 적용 범위에 의존하지 않고 읽기 제한을 유지합니다.

동반 script는 같은 hook을 설치하며 hook은 이미 서비스 디렉터리 읽기를 거부합니다. script는 독립적인 두 번째 계층으로 `disallowed_tools` 규칙도 추가하여 env와 세션 파일이 있는 해당 디렉터리의 `Read` tool을 차단합니다. 규칙은 `service_dir` parameter가 전달될 때만 설정되고 notebook 호출은 설정하지 않습니다.

같은 보호 조치는 [prompt injection 방어](https://code.claude.com/docs/en/security#protect-against-prompt-injection)에서 다루는 prompt injection에도 대응합니다. `tools`를 `Read`, `Glob`, `Grep`로 제한하면 세션에 shell과 network tool이 없고 읽은 내용이 갈 수 있는 곳은 응답 자체뿐입니다. 이 때문에 notebook의 정답 기준이 응답을 확인하고, `review.log`는 검토 repository와 같은 취급을 받습니다.


In [6]:
from typing import Any

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    HookContext,
    HookMatcher,
    ResultError,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    query,
)

READ_ONLY_TOOLS = ["Read", "Glob", "Grep"]
MODEL = "claude-sonnet-5"
FIRST_RUN_BUDGET_USD = 0.25
FOLLOW_UP_BUDGET_USD = 0.20
FIRST_RUN_MAX_TURNS = 20
FOLLOW_UP_MAX_TURNS = 14
TEXT_LOG_CHARS = 400


def clip(text: Any) -> str:
    """text를 한도가 있는 log 한 줄로 펼치며 잘림이 있으면 표시합니다.

    발견은 자유로운 text로 도착하고 다른 tool이 grep하는 log에 기록됩니다.
    출력할 수 없는 문자를 버리고 whitespace를 합쳐 필드마다 자체 log 한 줄에
    유지합니다.
    """
    printable = "".join(char if char.isprintable() else " " for char in str(text))
    flat = " ".join(printable.split())
    return flat if len(flat) <= TEXT_LOG_CHARS else flat[:TEXT_LOG_CHARS] + "..."


REVIEWER_SYSTEM_PROMPT = (
    "You are a repository review agent running on a schedule with nobody "
    "watching. Inspect the files with your read-only tools, keep the review "
    "short, and answer with the JSON object the caller's schema describes."
)


def deny_reads_outside_repo(repo: Path):
    """경로가 repo 밖으로 결정되는 모든 Read, Grep, Glob 호출을 거부합니다.

    해당 tool의 Read 규칙 적용에 의존하지 않고 Grep과 Glob을 결정적으로 제한합니다.
    후보 경로는 symlink를 해석하므로 repository 내 link를 통한 Read도 밖을 가리키면
    거부됩니다. 절대 Glob 패턴, `..` segment가 있는 패턴, segment 확인이 판단할 수 없는
    brace 문법의 패턴은 search root 안에 머무를 것이라고 믿는 대신 거부합니다.
    """

    def denial(reason: str) -> dict[str, Any]:
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": reason,
            }
        }

    async def deny(
        input_data: dict[str, Any], tool_use_id: str | None, context: HookContext
    ) -> dict[str, Any]:
        tool_input = input_data.get("tool_input") or {}
        raw = tool_input.get("file_path") or tool_input.get("path")
        if raw:
            candidate = Path(raw).expanduser()
            if not candidate.is_absolute():
                candidate = repo / candidate
            candidate = candidate.resolve()
            if not candidate.is_relative_to(repo):
                return denial(f"path outside the repository under review: {candidate}")
        if input_data.get("tool_name") == "Glob":
            pattern = tool_input.get("pattern")
            if isinstance(pattern, str) and (
                pattern.startswith(("/", "~")) or "{" in pattern or ".." in pattern.split("/")
            ):
                return denial(f"glob pattern may leave the repository under review: {pattern}")
        return {}

    return deny


def review_options(
    repo: Path,
    schema: dict[str, Any],
    max_turns: int,
    resume_session_id: str | None = None,
    service_dir: Path | None = None,
) -> ClaudeAgentOptions:
    """정기 review 한 번의 옵션을 만듭니다."""
    options = ClaudeAgentOptions(
        # agent는 검토 repository 안에서 작업합니다.
        cwd=str(repo),
        system_prompt=REVIEWER_SYSTEM_PROMPT,
        tools=READ_ONLY_TOOLS,
        # // 기준 형태는 의도적입니다. 선행 slash 하나면
        # filesystem 루트가 아니라 설정 source를 기준으로 하므로
        # 아무것도 matching되지 않아 dontAsk에서 모든 읽기가 거부됩니다.
        # repo는 절대 경로이므로 생성된 규칙은 이중 slash를 유지합니다.
        allowed_tools=[f"Read(/{repo}/**)", "Glob", "Grep"],
        permission_mode="dontAsk",
        model=MODEL,
        # 기기 및 계정에 설정된 MCP server를 세션에서
        # 제외합니다.
        strict_mcp_config=True,
        max_turns=max_turns,
        max_budget_usd=FOLLOW_UP_BUDGET_USD if resume_session_id else FIRST_RUN_BUDGET_USD,
        hooks={
            "PreToolUse": [
                HookMatcher(matcher="Read|Grep|Glob", hooks=[deny_reads_outside_repo(repo)])
            ]
        },
        # laptop과 cron 실행 기기에서 같은 동작.
        setting_sources=[],
        output_format={"type": "json_schema", "schema": schema},
        # 최초 실행은 None, 후속 실행은 저장된 세션 id.
        resume=resume_session_id,
    )
    if service_dir is not None:
        # 이미 읽기를 거부하는 hook 옆의 독립적인 두 번째 계층:
        # env 및 세션 파일이 있는 서비스 디렉터리의 Read tool을
        # 차단합니다.
        options.disallowed_tools = [f"Read(/{service_dir}/**)"]
    return options

## fallback이 있는 응답 읽기

응답은 이미 review schema로 검증된 상태로 도착합니다. 아래 두 reader인 `verdict_of`와 `finding_ids`는 코드가 동작에 사용하는 `verdict`와 `findings` 두 필드를 사용할 때 다시 확인합니다. 각 reader는 필요한 필드 형태를 matching하고 아니면 고정 기본값으로 fallback합니다. 이 셀은 runner가 채우는 container인 `RunOutcome`과 연속성 확인용 목록 reader인 `string_items`도 정의합니다.

누락되거나 알 수 없는 판정은 주의를 끄는 상태인 `concerns`로 읽습니다. 다른 형태의 발견 목록에서는 부분적인 추측 대신 id 없음으로 반환합니다. 마지막에 잘못된 입력으로 두 기본값을 실행하므로 아래 출력에서 fallback을 보여 줍니다.


In [7]:
from dataclasses import dataclass, field


@dataclass
class RunOutcome:
    """정기 실행 한 번이 보고한 내용."""

    session_id: str | None = None
    subtype: str = "no_result"
    num_turns: int = 0
    total_cost_usd: float | None = None
    denials: int = 0
    payload: dict[str, Any] = field(default_factory=dict)
    tools_attempted: list[str] = field(default_factory=list)


def sequence_of(value: Any) -> list[Any]:
    """목록 형태 필드의 항목을 반환하며 목록이 아니면 아무것도 반환하지 않습니다."""
    match value:
        case [*items]:
            return items
        case _:
            return []


def verdict_of(payload: dict[str, Any]) -> Verdict:
    """structured reply에서 판정을 읽고 필드가 누락되거나 알 수 없으면
    concerns로 fallback합니다."""
    match payload:
        case {"verdict": str(value)} if value in VERDICT_VALUES:
            return Verdict(value)
        case _:
            return Verdict.CONCERNS


def finding_ids(payload: dict[str, Any]) -> list[str]:
    """structured reply에서 발견 id를 모읍니다."""
    ids: list[str] = []
    for finding in sequence_of(payload.get("findings")):
        match finding:
            case {"id": str(finding_id)}:
                ids.append(finding_id)
    return ids


def string_items(value: Any) -> list[str]:
    """목록 형태 필드에서 일반 문자열을 모읍니다."""
    return [item for item in sequence_of(value) if isinstance(item, str)]


print("fallbacks:", verdict_of({}), finding_ids({"findings": "not-a-list"}))

fallbacks: concerns []


## review runner 만들기

schema, 옵션, reader가 runner에서 결합됩니다. runner는 message를 streaming하고 reviewer가 하는 말을 출력하고 사용하는 tool을 나열하며, 최종 `ResultMessage`의 `session_id`와 `structured_output`을 보관합니다.

각 실행 마지막 보고서는 판정을 출력한 뒤 응답 reader로 읽은 각 발견을 별도 줄에 출력합니다. block별 출력은 제한되며 잘린 줄에 보이는 표시가 있어 긴 응답 하나가 정기 log를 채우는 것을 막습니다. structured output 실행은 보통 설명문을 거의 내지 않으며 tool 호출 전에 한두 줄 설명할 수 있습니다. 기록된 실행에서는 두 주기 모두 tool 호출만 streaming합니다. 어느 경우든 log의 발견은 streaming text 대신 응답 payload에서 나옵니다.

같은 runner가 notebook 옆의 [`scheduled_review.py`](scheduled_review.py)로 제공되며 reviewer를 정기 실행할 때 deploy하는 동반 script입니다.

실패한 실행은 0이 아닌 종료 코드로 끝나며 완료 줄을 전혀 출력하지 않습니다. 두 실패 유형 모두 같습니다.

* **한도 초과**: `max_turns`나 `max_budget_usd`는 terminal 오류 결과와 0이 아닌 CLI 종료 코드로 실행을 끝냅니다. SDK는 결과의 subtype을 가진 `ResultError`를 발생시켜 코드에 제공합니다.
* **terminal 결과 없이 실패**: 도중에 종료된 CLI process나 전혀 열리지 않는 연결은 자체 exception 타입을 발생시키며 타입이 지정된 catch를 벗어나므로 실패 줄도 출력하지 않습니다.

각 review 실행은 `run_review` 호출 주변에서 `ResultError`를 잡고 `REVIEW-CYCLE-INCOMPLETE` 줄 하나를 출력한 뒤 다시 발생시킵니다. notebook은 성공 줄로 계속하지 않고 실패에서 멈춥니다. 실패 경로 증명의 유도된 실행은 다시 발생시키지 않고 `ResultError`를 잡습니다. 실패를 일으키는 것이 해당 셀의 목적입니다.


In [8]:
async def run_review(
    *,
    repo: Path,
    label: str,
    prompt: str,
    schema: dict[str, Any],
    max_turns: int,
    resume_session_id: str | None = None,
    service_dir: Path | None = None,
) -> RunOutcome:
    """review 한 번을 실행하고 structured result를 반환합니다."""
    options = review_options(repo, schema, max_turns, resume_session_id, service_dir)
    outcome = RunOutcome()
    async for message in query(prompt=prompt, options=options):
        match message:
            case AssistantMessage(content=blocks):
                for block in blocks:
                    match block:
                        case TextBlock(text=text):
                            line = text.strip()
                            if line:
                                # 정기 log가 block별로 받을 내용을 제한합니다.
                                print(f"[{label}] {clip(line)}")
                        case ToolUseBlock(name=tool_name):
                            outcome.tools_attempted.append(tool_name)
            case ResultMessage() as result:
                outcome.session_id = result.session_id
                outcome.subtype = result.subtype
                outcome.num_turns = result.num_turns
                outcome.total_cost_usd = result.total_cost_usd
                outcome.denials = len(result.permission_denials or [])
                if isinstance(result.structured_output, dict):
                    outcome.payload = result.structured_output
    return outcome


def report(label: str, outcome: RunOutcome) -> Verdict:
    """실행 한 번의 scheduler용 줄을 출력하고 판정을 반환합니다."""
    cost = f"{outcome.total_cost_usd:.4f}" if outcome.total_cost_usd is not None else "n/a"
    print(
        f"{label} session={outcome.session_id} subtype={outcome.subtype} "
        f"turns={outcome.num_turns} denials={outcome.denials} "
        f"cost_usd={cost} "
        f"tools_attempted={','.join(dict.fromkeys(outcome.tools_attempted)) or 'none'}"
    )
    verdict = verdict_of(outcome.payload)
    print(f"VERDICT: {verdict}")
    for finding in sequence_of(outcome.payload.get("findings")):
        match finding:
            case {"id": str(finding_id), "file": str(file_path), "summary": str(summary)}:
                print(f"  {clip(finding_id)} {clip(file_path)}: {clip(summary)}")
    return verdict

## 최초 review 실행하기

두 실제 실행이 간격을 없앤 일정의 첫 두 주기로 여기서 시작합니다. 첫 review는 새로 실행됩니다. `resume`이 없으면 전체 repository를 읽고 첫 review schema로 답합니다. 판정, 발견 id 집합, 다음 실행이 재개할 `session_id`와 함께 끝납니다.

**예상 출력:** `subtype=success`인 `RUN-1` 요약 줄 다음에 `VERDICT: concerns`와 심어 둔 두 결함을 다루는 발견 줄이 나옵니다. 결함 하나가 여러 발견으로 나뉠 수 있지만 기록된 실행에서는 결함당 하나를 출력합니다. 요약의 세션 id가 실행 2에서 재개하는 것입니다.

요약 줄의 0이 아닌 `denials` 수는 읽기 제한이 작동하는 것입니다. hook은 `/app/config.py`처럼 루트 기준 `Read` 시도를 거부하고 reviewer는 repository 내부 경로로 재시도합니다. `**/*.{py,md}` 같은 brace expansion Glob 패턴도 같은 수에 포함됩니다. 거부가 나타나는지는 처음 시도하는 경로와 패턴에 달려 있습니다.

거부는 turn budget 안에서 재시도 비용을 씁니다. repository review가 brace 패턴에 의존한다면 통째로 거부하는 대신 hook이 확장하고 각 결과에 같은 확인을 하도록 조정하세요.


In [9]:
import uuid

FIRST_REVIEW_ID = f"review-{uuid.uuid4().hex[:8]}"
FIRST_PROMPT = (
    f"Scheduled review {FIRST_REVIEW_ID} of the repository in the current "
    "directory. "
    "Read the source files and report correctness or security problems you can "
    "point at a specific file and function. Give each finding a summary of "
    "one or two sentences and an id of the form "
    'F1, F2, and so on. Use the verdict "concerns" when you report at least one '
    'finding and "ok" when the repository looks clean. Echo '
    f"{FIRST_REVIEW_ID} back as review_id."
)

try:
    first = await run_review(
        repo=REPO_DIR,
        label="run-1",
        prompt=FIRST_PROMPT,
        schema=FIRST_REVIEW_SCHEMA,
        max_turns=FIRST_RUN_MAX_TURNS,
    )
except ResultError as exc:
    cost = (exc.data or {}).get("total_cost_usd")
    cost_str = f"{cost:.4f}" if isinstance(cost, (int, float)) else "n/a"
    print(
        f"REVIEW-CYCLE-INCOMPLETE stage=run-1 subtype={exc.subtype} "
        f"reason={exc.terminal_reason} cost_usd={cost_str}"
    )
    raise

if first.subtype != "success" or first.session_id is None or not first.payload:
    print(f"REVIEW-CYCLE-INCOMPLETE stage=run-1 subtype={first.subtype}")
    raise RuntimeError("run 1 did not complete; there is no session to resume")
first_verdict = report("RUN-1", first)

RUN-1 session=2d83a97b-27ff-403e-a067-2fb5b6474d9b subtype=success turns=10 denials=3 cost_usd=0.0278 tools_attempted=Glob,Read,StructuredOutput
VERDICT: concerns
  F1 app/math_utils.py: divide() does not guard against denominator being zero, and average() calls divide(sum(values), len(values)) which raises ZeroDivisionError when values is an empty list.
  F2 app/config.py: load_config() copies the entire process environment (os.environ) into the config dict and prints it via print(), which can leak secrets/credentials (API keys, tokens) into logs.


## 주기 사이에 repository 변경하기

deployment에서는 주기 사이에 들어온 변경이 보고된 발견을 수정하거나 아직 보지 못한 문제를 도입합니다. 아래 코드는 후속 실행 전에 각 종류의 변경을 하나씩 수행합니다.

후속 결과에서 resolved가 되어야 하는 `app/math_utils.py`의 zero guard를 수정합니다. 그리고 후속 실행이 새로 잡아야 하는 문제인 무제한 retry 루프의 새 파일 `app/retry.py`를 만듭니다. 새 문제는 실행 1이 읽지 않은 파일에 있습니다. 이를 잡으면 이미 아는 파일로 답하는 대신 현재 repository를 살폈음을 증명합니다.


In [10]:
MATH_UTILS_FIXED = """def divide(numerator, denominator):
    if denominator == 0:
        raise ValueError("denominator must be non-zero")
    return numerator / denominator


def average(values):
    if not values:
        return 0.0
    return divide(sum(values), len(values))
"""

RETRY_PY = """import time


def fetch_with_retry(fetch):
    while True:
        try:
            return fetch()
        except ConnectionError:
            time.sleep(1)
"""

(REPO_DIR / "app/math_utils.py").write_text(MATH_UTILS_FIXED, encoding="utf-8")
(REPO_DIR / "app/retry.py").write_text(RETRY_PY, encoding="utf-8")
changed = sorted(str(path.relative_to(REPO_DIR)) for path in REPO_DIR.rglob("*") if path.is_file())
print("repository now:", changed)

repository now: ['README.md', 'app/config.py', 'app/math_utils.py', 'app/retry.py']


## 재개 review 실행하기

다음 정기 실행은 첫 `session_id`를 `resume`으로 전달하고 후속 schema를 요청합니다. prompt의 첫 지시는 repository 파일을 다시 나열하는 것입니다. 없으면 재개 agent는 실행 1에서 읽은 파일로 답하고 `app/retry.py`가 새로 생겼음을 전혀 알아차리지 못할 수 있습니다. 아래 코드는 실행 1과 응답을 대조하여 세 필드의 연속성 증명을 담은 `RESUME-LINK` 줄을 출력합니다.

* `same_session`: 두 실행이 세션 하나를 공유하여 재개 자체가 작동했음을 보여 줍니다.
* `prior_review_id_echoed`: 후속 review가 첫 review의 id를 명시했습니다.
* `recalled_findings`: 첫 실행의 발견 id 중 후속 review가 돌려준 수입니다.

셋 다 성립하면 reviewer는 이미 한 review를 기반으로 답한 것입니다. 하나라도 실패하면 코드가 별도 줄에 `RESUME-LINK-BROKEN` 표시를 출력합니다. 응답은 더 이상 적용되지 않는 이전 발견 id인 `resolved`도 나열합니다.

**예상 출력:**

* `same_session=True`, `prior_review_id_echoed=True`, 전체 `recalled_findings` 수를 가진 `RESUME-LINK` 줄
* zero guard 발견 id를 가진 `resolved` 목록
* config 유출을 유지하고 새 retry 버그를 추가한 발견
* 주기를 마치는 `REVIEW-CYCLE-COMPLETE` 줄


In [11]:
SECOND_REVIEW_ID = f"review-{uuid.uuid4().hex[:8]}"
FOLLOW_UP_PROMPT = (
    f"Scheduled review {SECOND_REVIEW_ID} of the same repository, one cycle "
    "later. List the repository's files again, then read the source files as "
    "they stand now. Answer from the most recent review you did in this "
    "conversation: put its review id in previous_review_id and the ids of the "
    "findings it reported in previous_finding_ids. Report the findings that "
    "still apply, keeping the ids you gave them earlier, list in resolved the "
    "ids of previous findings that no longer apply, and report anything newly "
    "wrong as a new finding with a summary of one or two sentences. Echo "
    f"{SECOND_REVIEW_ID} back as review_id."
)

try:
    second = await run_review(
        repo=REPO_DIR,
        label="run-2",
        prompt=FOLLOW_UP_PROMPT,
        schema=FOLLOW_UP_REVIEW_SCHEMA,
        max_turns=FOLLOW_UP_MAX_TURNS,
        resume_session_id=first.session_id,
    )
except ResultError as exc:
    cost = (exc.data or {}).get("total_cost_usd")
    cost_str = f"{cost:.4f}" if isinstance(cost, (int, float)) else "n/a"
    print(
        f"REVIEW-CYCLE-INCOMPLETE stage=run-2 subtype={exc.subtype} "
        f"reason={exc.terminal_reason} cost_usd={cost_str}"
    )
    raise

if second.subtype != "success" or second.session_id is None or not second.payload:
    print(f"REVIEW-CYCLE-INCOMPLETE stage=run-2 subtype={second.subtype}")
    raise RuntimeError("run 2 did not complete")
second_verdict = report("RUN-2", second)

prior_ids = finding_ids(first.payload)
recalled_ids = string_items(second.payload.get("previous_finding_ids"))
matched = sorted(set(recalled_ids) & set(prior_ids))
echoed = second.payload.get("previous_review_id") == FIRST_REVIEW_ID
print(
    f"RESUME-LINK run1_session={first.session_id} "
    f"run2_session={second.session_id} "
    f"same_session={second.session_id == first.session_id} "
    f"prior_review_id_echoed={echoed} "
    f"recalled_findings={len(matched)}/{len(set(prior_ids))}"
)
print(f"resolved={[clip(i) for i in string_items(second.payload.get('resolved'))]}")
if second.session_id != first.session_id or not echoed or (prior_ids and not matched):
    # script의 표시처럼 의도적으로 완료된 실행에서 출력합니다.
    print(
        f"RESUME-LINK-BROKEN same_session={second.session_id == first.session_id} "
        f"recalled_findings={len(matched)}/{len(set(prior_ids))}"
    )
print(f"REVIEW-CYCLE-COMPLETE runs=2 verdicts={first_verdict},{second_verdict}")

RUN-2 session=2d83a97b-27ff-403e-a067-2fb5b6474d9b subtype=success turns=7 denials=0 cost_usd=0.0339 tools_attempted=Glob,Read,StructuredOutput
VERDICT: concerns
  F2 app/config.py: load_config() still copies the entire process environment (os.environ) into the config dict and prints it via print(), which can leak secrets/credentials into logs.
  F3 app/retry.py: fetch_with_retry() retries in an unbounded infinite loop with a fixed 1-second sleep and no max attempt limit or exponential backoff, risking indefinite hangs/resource exhaustion if the underlying fetch keeps raising ConnectionError; other exceptions are also not handled.
RESUME-LINK run1_session=2d83a97b-27ff-403e-a067-2fb5b6474d9b run2_session=2d83a97b-27ff-403e-a067-2fb5b6474d9b same_session=True prior_review_id_echoed=True recalled_findings=2/2
resolved=['F1']
REVIEW-CYCLE-COMPLETE runs=2 verdicts=concerns,concerns


## 심어 둔 변경과 발견 대조하기

repository의 버그와 주기 간 변경을 미리 알므로 두 실행의 정답 기준으로 사용합니다.

### 실행 1의 보고 확인하기

실행 1은 적어도 심어 둔 두 버그를 보고합니다.

* `app/config.py`: loader가 전체 environment를 config에 복사하여 출력
* `app/math_utils.py`: `divide`에 zero guard가 없어 `average([])`가 `ZeroDivisionError` 발생

`config["environment"] = dict(os.environ)`은 무해해 보이는 한 줄입니다. 다음 줄의 `print`가 공개하고 deployment가 environment variable에 secret을 두므로 유출이 존재합니다. 보고했다면 reviewer가 파일 안의 세 사실을 연결한 것입니다. zero guard는 세심히 읽는지만 시험합니다.

### 후속 review의 차이 확인하기

후속 보고는 주기 간 변경을 세 부분으로 나눕니다.

* config 유출은 이전 id로 `findings`에 남음
* 수정된 zero guard의 id는 `resolved`로 이동
* `app/retry.py`의 무제한 retry는 새 발견으로 도착

`prior_review_id_echoed=True`는 재개 reviewer가 실행 1에서 받은 id를 명시했음을 뜻하며 `recalled_findings`는 실행 1의 발견 id가 몇 개 돌아왔는지 셉니다.

### 실행 간 차이 예상하기

model 출력은 실행마다 다릅니다. reviewer는 다음을 할 수 있습니다.

* 발견을 다르게 표현
* 다른 id 부여
* 심어 둔 버그 하나를 여러 발견으로 나누기(zero guard가 하나 또는 별도 `divide`와 `average` 발견으로 도착 가능)
* 추가적인 낮은 심각도의 관찰 보고

reviewer가 어떻게 나누었든 `resolved`의 id는 변경으로 수정한 내용을 따릅니다. 표현과 관계없이 올바른 첫 실행은 두 버그를 보고하고 올바른 후속 실행은 세 부분 차이를 보고하며 `RESUME-LINK` 필드가 연속성 증명을 담습니다.

`demo_repo/`의 다른 내용을 바꾸고 후속 review를 다시 실행하여 차이가 변하는 것을 보세요. notebook은 baseline을 실행 1로 고정하므로 반복 후속 실행은 재개가 작동했어도 `prior_review_id_echoed=False`와 `RESUME-LINK-BROKEN` 표시를 출력합니다. 동반 script는 매 실행 후 새 baseline을 저장하여 반복 후속 실행도 연결을 유지합니다.


## 실패 경로 증명하기

runner 절은 실패한 실행의 모습을 설명합니다. 아래 셀은 전체 review가 들어갈 수 없는 `max_turns=1` 상한으로 같은 repository의 새 review를 실행하여 의도적으로 실패를 일으킵니다. 실행은 terminal 오류 결과로 끝나고 SDK가 `ResultError`를 발생시키며 `except` block은 결과의 `subtype`과 `terminal_reason`을 담은 실패 줄을 출력합니다.

**예상 출력:** `stage=induced subtype=error_max_turns reason=max_turns`와 실패 실행이 소비한 비용을 담은 `REVIEW-CYCLE-INCOMPLETE` 줄, 완료 줄 없음.


In [12]:
try:
    await run_review(
        repo=REPO_DIR,
        label="induced",
        prompt=FIRST_PROMPT,
        schema=FIRST_REVIEW_SCHEMA,
        max_turns=1,
    )
except ResultError as exc:
    cost = (exc.data or {}).get("total_cost_usd")
    cost_str = f"{cost:.4f}" if isinstance(cost, (int, float)) else "n/a"
    print(
        f"REVIEW-CYCLE-INCOMPLETE stage=induced subtype={exc.subtype} "
        f"reason={exc.terminal_reason} cost_usd={cost_str}"
    )
else:
    raise RuntimeError("the run completed under max_turns=1")

REVIEW-CYCLE-INCOMPLETE stage=induced subtype=error_max_turns reason=max_turns cost_usd=0.0032


## reviewer를 정기 실행하기

위 두 주기는 수동 실행했습니다. 일정에서는 각 주기가 notebook 옆 동반 script [`scheduled_review.py`](scheduled_review.py)의 한 번 호출입니다. 자체 서비스 디렉터리에서 실행하고 검토 repository를 인자로 받으며 같은 schema와 응답 reader를 가집니다.

notebook 주기의 `REVIEW-CYCLE` 대신 script 표시는 `REVIEW-RUN`이며 정기 log에는 notebook 표시가 나오지 않습니다. 서비스 디렉터리에 `.last_review_session` 세션 파일이 없으면 새로 검토하고 있으면 재개합니다. `read_state` 함수는 여기서 notebook 변수가 대신했던 저장 상태를 관리합니다.

일정을 설정하기 전에 script를 한 번 읽으세요.

지속되는 host에서 reviewer를 실행하세요. review에 필요한 것만 접근할 수 있는 사용자를 작업에 지정하고 아래 모든 단계를 해당 사용자로 실행하세요. 재개에는 두 로컬 파일이 주기 사이에 유지되어야 합니다.

* script가 자체 서비스 디렉터리의 옆에 유지하는 세션 파일 `.last_review_session`
* SDK가 `~/.claude/projects/` 또는 `CLAUDE_CONFIG_DIR` 설정 시 그 안의 `projects/`에 쓰는 세션 transcript

서비스 디렉터리는 검토 repository 밖에 두세요. 그러면 clean checkout이나 branch 전환으로 세션 파일이 삭제되지 않고 reviewer가 읽는 파일에도 나타나지 않습니다. 이 예제 자체 디렉터리의 시험 실행 `python scheduled_review.py demo_repo`는 시작 시 `reason=repository-inside-service-directory` 사용법 줄과 함께 거부됩니다. repository 안의 서비스 디렉터리도 같은 방식으로 `reason=service-directory-inside-repository`와 함께 거부됩니다.

### script 일정 설정하기

1. 자체 environment를 가진 서비스 디렉터리를 만듭니다. notebook의 `%pip` 설치는 자체 environment에만 적용되고 cron은 명령이 지정한 interpreter를 실행합니다. 서비스 디렉터리는 SDK가 설치된 자체 virtual environment를 가집니다.

   ```bash
   sudo mkdir -p /srv/reviewer && sudo chown $USER /srv/reviewer && cd /srv/reviewer
   python3 -m venv .venv
   .venv/bin/pip install "claude-agent-sdk>=0.2.140"
   cp /path/to/scheduled_review.py .
   ```

   사전 요구 사항의 하한인 3.11 이상의 `python3`를 사용하세요. 더 오래된 interpreter는 script의 첫 import에서 실패합니다.

2. cron 작업이 source하는 environment 파일 `/srv/reviewer/reviewer.env`를 만듭니다. cron은 거의 없는 environment로 시작하므로 scheduler가 제공해야 합니다. 단순 `KEY=value`는 Python process가 상속하지 않는 shell variable을 설정하므로 `export` 줄을 쓰세요. Cron은 `PATH`를 최소화합니다. 필요한 binary가 있는 디렉터리를 포함하도록 설정하세요.

   ```bash
   # /srv/reviewer/reviewer.env, cron 작업이 source함
   export PATH=/usr/local/bin:$PATH
   export ANTHROPIC_API_KEY=your_key_here
   ```

   key를 넣기 전에 `touch /srv/reviewer/reviewer.env && chmod 600 /srv/reviewer/reviewer.env`로 빈 파일을 만들고 권한을 제한한 뒤 위 줄을 추가하세요. 기본적으로 script는 자신에게 서비스 디렉터리의 `Read` 접근도 거부하여 key 파일이 발견 내용에 들어가지 않게 합니다. 같은 거부는 서비스 디렉터리 안의 repository review도 보지 못하게 하므로 script가 그 구조를 아예 거부합니다. crontab을 나열할 수 있는 사람은 읽을 수 있으므로 key를 crontab 줄 자체에 넣지 마세요.

3. 서비스 디렉터리에서 검토할 repository를 가리키도록 수동으로 한 번 실행합니다.

   ```bash
   cd /srv/reviewer && . ./reviewer.env && .venv/bin/python scheduled_review.py /path/to/your/repo
   ```

   첫 실행은 새로 검토하고 `REVIEW-RUN-COMPLETE: cold`를 출력하며 모든 정기 실행이 재개할 `.last_review_session`과 세션 transcript를 생성합니다. `error_max_budget_usd`로 1을 반환하면 일정 설정 전에 `FIRST_RUN_BUDGET_USD`를 올리세요. budget에 들어가지 못하는 최초 실행은 baseline을 만들지 못하므로 모든 정기 주기가 같은 실패를 반복합니다.

4. `crontab -e`로 매일 02:00 review 항목을 추가합니다. transcript가 사용자 home에 있으므로 3단계를 수행한 같은 사용자로 설정하세요.

   ```bash
   0 2 * * * cd /srv/reviewer && . ./reviewer.env && flock -n -E 99 .review.lock .venv/bin/python scheduled_review.py /path/to/your/repo >> /srv/reviewer/review.log 2>&1 || echo "REVIEW-RUN-EXIT: $?" >> /srv/reviewer/review.log
   ```

   항목의 뒷부분은 실패한 호출의 종료 status를 log 줄로 바꿉니다. 두 redirect는 절대 경로로 log를 지정하며 cron의 시작 디렉터리 기준 파일에 표시가 남지 않습니다.

   `flock -n`은 이전 실행이 lock을 보유 중이면 호출을 건너뛰며 `-E 99`는 별도 종료 코드를 부여합니다. 건너뛴 실행은 `REVIEW-RUN-EXIT: 99`를 기록하여 실패한 실행처럼 보이지 않습니다. `flock`은 Linux host의 util-linux에서 제공합니다. macOS에는 기본 flock이 없고 Homebrew의 [flock formula](https://formulae.brew.sh/formula/flock)가 제공합니다.

   발견은 repository 내용을 인용하므로 `review.log`를 검토 repository와 같이 취급하세요. 다른 서비스 log처럼 rotate하세요. 다음 절의 `REVIEW-RUN-EXIT` 행이 읽는 방법을 보여 줍니다.


### 출력 줄 읽기

scheduler는 다음 줄을 읽습니다.

| 줄 | 발생 시점 | 종료 코드 | 알림 |
|---|---|---|---|
| `VERDICT: ok` 또는 `VERDICT: concerns` | 완료된 review가 structured reply 판정을 보고. 실패한 실행은 출력하지 않음 | 0 | `concerns`에 알림 |
| `RESUME-LINK ...` | 모든 재개 실행. `same_session=True`와 안정적인 `recalled_findings` 수는 연속성이 작동함을 뜻하며 `current_findings`는 후속 review 자체 발견 수 | 0 | 자체 알림에서 `recalled_findings` 한도 설정. 점차 떨어지는 수는 표시를 출력하지 않으며 정기 세션 reset 절이 설명하는 reset 신호 |
| `RESUME-LINK-BROKEN ...` | 세션 유실, echo 유실, recall 0으로 연결이 완전히 끊김. grep할 수 있도록 표시가 줄 시작 | 0 | 알림. 다음 주기용 새 baseline은 이미 저장됨 |
| `REVIEW-RUN-COMPLETE: cold` 또는 `: resumed` | review 성공이 검증됨 | 0 | 없음. 존재하면 주기 완료 |
| `REVIEW-RUN-INCOMPLETE ...` | 보통 한도 초과의 terminal 오류 결과로 종료. `subtype`이 종료 원인을 명시하고 한도 및 schema 실패는 `reason` 추가. 세션 crash 후 `cost_usd=0.0000`이면 비용 0이 아니라 알 수 없음 | 1 | 알림. 한도 실패는 세션 파일을 유지하며 config 변경으로 수정 |
| `REVIEW-RUN-INCOMPLETE ... subtype=error_max_structured_output_retries` | 응답이 반복적으로 schema 검증 실패, 또는 [model fallback](https://code.claude.com/docs/en/model-config#automatic-model-fallback)이 완료 응답을 철회했는데 대체할 재시도가 없음. 둘 다 세션 문제가 아님 | 1 | 알림. 세션 파일 유지 |
| `REVIEW-RUN-INCOMPLETE stage=usage ...` | repository 인자 누락처럼 호출 자체가 잘못됨 | 2 | 알림. crontab 항목 수정 |
| `SESSION-FILE-CLEARED: next run reviews cold` | 재개 실행이 `subtype=error_during_execution`으로 실패. 보통 저장된 세션을 재개하지 못함. 일시적 연결 실패도 해당하며 최초 재구축 한 번으로 자체 복구 | 1 | 실패 알림 외 없음. 다음 주기는 baseline을 새로 구축 |
| `SESSION-STATE-NOT-SAVED ...` | review 완료 후 보통 디스크가 가득 찼거나 read-only여서 상태를 기록하지 못함. 위 발견은 유효함. 뒤의 `SESSION-FILE-CLEARED`는 다음 실행이 새로 구축함을, `SESSION-FILE-STALE`는 이전 세션을 재개함을 뜻함 | 1 | 알림. 디스크 수정. review 자체는 재실행 불필요 |
| `REVIEW-RUN-EXIT: <code>` | crontab 항목 뒷부분이 겹쳐서 건너뛴 실행의 `99`를 포함한 0이 아닌 종료 코드 기록 | n/a | 99 이외 모든 코드에 알림 |
| 줄이 전혀 없음 | process 종료나 열리지 않는 연결처럼 terminal 결과 전에 실행 중단 | 0이 아님 | `99`로 설명되지 않는 완료 줄 누락에 알림 |

유도된 `max_turns=1` 실행은 grep할 prefix인 `REVIEW-RUN-INCOMPLETE stage=cold subtype=error_max_turns reason=max_turns`로 시작하고 실행의 `cost_usd`로 끝나는 줄을 출력합니다. 실패 경로 증명 절이 notebook에서 같은 terminal 결과를 유발했습니다.

성공 주기는 log에 `REVIEW-RUN-COMPLETE`를 남기고 실패 주기는 `REVIEW-RUN-EXIT`를 남깁니다. 99 이외 모든 `REVIEW-RUN-EXIT` 코드나 `99` 건너뛰기로 설명되지 않는 완료 줄 누락 주기에 알림을 설정하세요. 건너뛰기는 정상적인 중복이며 반복되면 실행이 일정 간격보다 오래 걸린다는 뜻입니다. 모든 표시의 grep을 줄 시작에 고정하세요. 발견 text는 한 줄씩 들여쓰여 잘리므로 표시는 항상 자체 줄을 시작합니다. `stage=resumed` 실패 뒤 `SESSION-FILE-CLEARED`가 있으면 저장된 세션이 더 이상 재개되지 않았다는 뜻이며, 보통 `~/.claude/projects/`의 transcript가 없어졌기 때문입니다.


### reviewer 맞춤 설정하기

* **prompt**: 새 dependency, 호환성을 깨는 API 변경, TODO 부채처럼 review가 추적할 대상에 후속 prompt를 맞추세요.
* **schema**: 발견별 severity처럼 알림에 필요한 필드를 추가하세요. 선언한 내용에 맞춰 검증된 응답이 돌아옵니다. script는 자체 schema 및 reader 복사본을 가집니다. 어느 쪽을 맞춤 설정하든 둘 다 변경하세요.
* **한도**: `max_turns`와 `max_budget_usd`는 script 위쪽에 있습니다. notebook 셀의 두 한도는 기록된 실행의 `cost_usd`보다 여유를 둔 파일 세 개 예제용입니다. script 기본값은 실제 repository의 여유를 둡니다. budget은 turn 상한처럼 나뉜 폭주 방지 장치입니다. baseline review는 `FIRST_RUN_BUDGET_USD`, 모든 후속 review는 `FOLLOW_UP_BUDGET_USD` 아래 실행됩니다. 일정이 반복하여 곱하는 것은 후속 상한입니다. 기본값에서 매일 실행 작업의 최악 한 달은 각 몇 센트인 기록된 실행과 비교하여 약 30 × $0.50 = $15에 가끔의 최초 재구축 비용입니다. 몇 주기의 요약 줄 `cost_usd`를 관찰하여 맞는 상한을 찾으세요. 수치는 청구 데이터 대신 client 측 추정치입니다. [비용과 usage 추적](https://code.claude.com/docs/en/agent-sdk/cost-tracking)이 차이를 다룹니다. [`effort` 옵션](https://code.claude.com/docs/en/agent-sdk/agent-loop#effort-level)은 또 하나의 비용 조절 수단입니다. 문서는 파일만 읽는 agent에 `"low"`를 권장하며 이 reviewer가 해당합니다.
* **model**: 비용과 기능을 조절하려면 `MODEL` alias를 바꾸세요. 고정으로 변경을 의도적으로 유지합니다.

## 정기적으로 세션 reset하기

장시간 유지되는 세션은 의도적으로 context를 모읍니다. 이 context는 비용도 늘리고 이전 결론에 reviewer를 묶을 수 있습니다. 매주 또는 repository 형태가 바뀔 때처럼 정기적으로 script 옆 세션 파일을 삭제하면 다음 실행이 새로 review하여 baseline을 재구축합니다.

후속 review의 `recalled_findings` 수가 운영 신호입니다. 수가 떨어지면 세션이 저하된 것이며 reset할 때입니다.


## 언제 재개하고 언제 새로 시작할지

정기 review 작업은 이전 세션을 재개하거나, 주기마다 새로 시작하거나, 이전 발견을 입력하여 새로 시작합니다. 아래 표는 선택별로 적합한 작업을 연결합니다.

| 선택 | 시점 |
|---|---|
| 호출별 새 세션 | 각 작업 단위를 자체적으로 평가합니다. pull request 통과를 결정하는 CI reviewer에 필요합니다. 이전 의견이나 이어받은 context가 현재 판정을 이끌지 않습니다. |
| 이전 발견을 입력한 새 세션 | 마지막 보고 내용만 필요합니다. 지난주 항목 중 아직 열린 것을 보고하는 TODO 부채 추적에 맞습니다. 응답의 발견 id가 비교에 필요한 모든 것을 담고, 매 주기 코드를 새로 평가하며, 주기 사이 저장 세션이 오래되거나 없어질 수 없습니다. |
| 재개 세션 | 마지막 검토에서 본 모든 내용과 비교합니다. 이전 이후 추가되거나 바뀐 package만 표시하는 매일 dependency audit은 reviewer가 읽은 전체 목록을 확인하며 대부분은 응답에 들어가지 않았습니다. 직접 전달하려면 관찰한 모든 것을 serialize해야 하며 재개가 이 작업을 줄입니다. |


## 정리하기

notebook은 두 주기 fixture와 주기 간 변경으로 `demo_repo/`에 파일 네 개를 썼습니다. 아래 코드는 디렉터리가 fixture를 포함하는지 확인한 뒤 제거합니다. 먼저 정답 기준의 차이 실험을 하려면 정리를 미루고, 나중에 파일이 필요하면 예제 repository 코드를 다시 실행하세요.

동반 script도 실행했다면 서비스 디렉터리의 `.last_review_session`도 삭제하세요. 실행은 `~/.claude/projects/` 아래 세션 transcript도 썼습니다. 제거하려면 해당 project 폴더를 삭제하세요.


In [13]:
import shutil

config_path = REPO_DIR / "app" / "config.py"
if config_path.exists() and config_path.read_text(encoding="utf-8") == CONFIG_PY:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    print("removed", REPO_DIR.name)
else:
    print("left", REPO_DIR.name, "in place: it does not hold the seeded fixture")

removed demo_repo


## 배운 내용

* **무인 agent 제한하기**: `tools`와 `allowed_tools`는 존재할 것과 prompt 없이 실행할 것을 결정하고 `max_turns` 및 `max_budget_usd`는 루프와 지출을 제한하므로 정기 실행의 잘못된 동작도 선택한 한도 안에 있습니다.
* **연속성 증명하기**: 이전 세션을 재개하는 agent는 발견한 것을 기억합니다. 매 실행이 이전 발견을 응답에서 되풀이하여 증명하게 하세요. 재개가 끊기면 발생 즉시 log에 나타납니다.
* **정기 reviewer 설정하기**: 무인 review는 프로그램에 보고하므로 출력은 scheduler가 grep할 수 있는 종료 코드와 고정 줄입니다. 이 reviewer에서는 repository에 대한 판정 줄, 성공 후에만 출력하는 완료 줄, 모든 실패를 표시하는 0이 아닌 종료 코드입니다.

## 자세히 알아보기

* [agent에서 structured output 얻기](https://code.claude.com/docs/en/agent-sdk/structured-outputs): 이 예제 응답의 `output_format` schema 및 `structured_output` 필드 전체 문서
* [세션 다루기](https://code.claude.com/docs/en/agent-sdk/sessions): 사용한 재개 외 세션 lifecycle, 두 경로로 세션 fork하기 포함
* [permission 설정하기](https://code.claude.com/docs/en/agent-sdk/permissions): `dontAsk`의 permission mode model과 interactive 사용용 mode
* [Security](https://code.claude.com/docs/en/security): 신뢰할 수 없는 내용 처리의 권장 방법을 포함한 Claude Code security 보호 조치
* [비용과 usage 추적](https://code.claude.com/docs/en/agent-sdk/cost-tracking): `max_budget_usd`와 요약 줄 비용 수치의 비용 인터페이스
